# Measure chunk latency

We measure amortized read cost by chunk size using the upstream profiler.

Run from this directory on the target SSD. See the [setup instructions](../../readme.md) for dependencies.

In [ ]:
import re
import sys
import numpy  as np
import pandas as pd 

from pathlib    import Path
from contextlib import redirect_stdout

In [ ]:
DIRECTORY = Path.cwd()
SUBMODULE = DIRECTORY / "../../vlm-flash"

# add submodule into system path
sys.path.append(str((SUBMODULE / "scripts").resolve()))
sys.path.append(str((SUBMODULE / "src")    .resolve()))

# ...to import profile_flash
import profile_flash as profiler

## Board state

Measure in the `MAXN_SUPER` power mode with CPU clocks pinned at their maximum.
Set both on the host before running. `jetson_clocks` pins each CPU clock by raising
its minimum to its maximum; run it after `nvpmodel`, and again after a reboot.

```bash
sudo nvpmodel -m 2
sudo jetson_clocks
```

In [ ]:
POWER_MODE = "MAXN_SUPER"

In [ ]:
def board_state():
    """Return the power mode and each CPU's (min, max, limit) clock in kHz."""

    # Power mode
    status = Path("/var/lib/nvpmodel/status").read_text()
    config = Path("/etc/nvpmodel.conf").read_text()

    index = int(re.search(r"pmode:(\d+)", status).group(1))
    names = re.findall(r"< POWER_MODEL ID=(\d+) NAME=(\S+) >", config)
    mode  = {int(i): name for i, name in names}[index]

    # CPU clocks
    clocks = []

    for cpu in sorted(Path("/sys/devices/system/cpu").glob("cpu[0-9]*/cpufreq")):
        clocks.append((
            int((cpu / "scaling_min_freq").read_text()),
            int((cpu / "scaling_max_freq").read_text()),
            int((cpu / "cpuinfo_max_freq").read_text()),
        ))

    return mode, clocks

In [ ]:
mode, clocks = board_state()

# MAXN_SUPER?
if mode != POWER_MODE:
    raise RuntimeError(f"Power mode is {mode}; run `sudo nvpmodel -m 2`.")

# pinned at maximum?
if not all(low == high == limit for low, high, limit in clocks):
    raise RuntimeError("CPU clocks are not pinned; run `sudo jetson_clocks`.")

mode, sorted(set(clocks))

## 1. Direct I/O

In [ ]:
# native module available?
if profiler.native() is None:
    raise RuntimeError(profiler.unavailable_reason())

In [ ]:
class DirectIOReader:

    def read_rows(self, *args, **kwargs):

        # read rows
        result = profiler.native().read_rows(*args, **kwargs)

        # not Direct I/O
        if not result[3]:
            raise RuntimeError("Direct I/O is required.")

        return result

reader = DirectIOReader()

## 2. Measurement

We measure each chunk size seven times using six threads and direct I/O. Size order is randomized within each repeat.

For chunk size $z$ in KiB and estimated batch throughput $\widehat\beta_b(z)$ in MiB/s, repeat $b$ gives

$$T_b(z)=\frac{1000z}{1024\widehat\beta_b(z)}\quad\mathrm{ms/chunk}.$$

Throughput counts logical bytes, excluding alignment padding. $T_b$ is amortized batch cost, not isolated-read latency.

In [ ]:
SIZES_KIB = [
    *range(  1, 321, 1),  #   1–320 KiB, step 1
    *range(324, 769, 4),  # 324–768 KiB, step 4
]

REPEATS  = 7    # Repetitions of the full size grid
ITERS    = 10   # Reads per test
WARMUP   = 3    # Warm-up reads
THREADS  = 6    # I/O threads
BLOB_MIB = 128  # File size (MiB)

num_rows = BLOB_MIB * 1024 // profiler.ROW_KB 

In [ ]:
# need this to shuffle sizes
rng     = np.random.default_rng(0)

# save latency of each sizes
records = []

In [ ]:
# check if another test is ongoing with identical name
with profiler.exclusive("happy_to_check_io_latency"): 

    # create or reuse the test file
    blob = str(DIRECTORY / "profile_blob.data")
    profiler.prepare_blob(blob, num_rows)

    # repeat measurements
    for repeat in range(REPEATS):

        print(f"Repeat {repeat + 1}/{REPEATS}", flush=True)

        # check latency for every size
        for order, size in enumerate(rng.permutation(SIZES_KIB).tolist()):

            # hide inner stdout from console
            with redirect_stdout(None):
                
                # measure
                runs = profiler.measure(
                    reader, 
                    blob, 
                    num_rows, 
                    [size],
                    ITERS, 
                    WARMUP, 
                    THREADS
                )[size]

                # Three-point mean; average the tail after the first absolute
                # slope below 5% of peak per MiB. Fallback: last 20% of points;
                # use the last point if fewer than three are available.
                rate = profiler.saturation_throughput(
                    runs, 
                    size
                ); 

            # save run results
            records.append(
                dict(
                    # repeat index
                    repeat           = repeat, 
                    
                    # order & size
                    order            = order, 
                    size_kib         = size,

                    # latency & throughput
                    latency_ms       = 1000 * size / (1024 * rate),
                    throughput_mib_s = rate,
                )
            )

print("Done.", flush=True)

## 3. Saved measurements

Write `chunk_latency_measurements.csv`, then run [01_02](01_02_chunk_latency.ipynb) to fit the cost models.

In [ ]:
data = pd.DataFrame(records)
data.head()

In [ ]:
data.to_csv(DIRECTORY / "chunk_latency_measurements.csv", index=False)